In [0]:
-- revenue_by_account_day: amounts by status and product type per account, currency and day
CREATE OR REFRESH MATERIALIZED VIEW gold.revenue_by_account_day (
  account_id STRING NOT NULL COMMENT 'Account key (FK to account_360.account_id). Part of the primary key.',
  account_name STRING COMMENT 'Company name of the account.',
  currency STRING NOT NULL COMMENT 'ISO currency code of all amounts in the row (not converted). Part of the primary key.',
  transaction_date DATE NOT NULL COMMENT 'Transaction calendar day. Part of the primary key.',
  month DATE COMMENT 'First day of the transaction_date month, for rolling up to months.',
  transaction_count BIGINT COMMENT 'Number of transactions in the period (all statuses).',
  completed_transaction_count BIGINT COMMENT 'Transactions with status Completed.',
  completed_amount DECIMAL(28,2) COMMENT 'Sum of Completed transaction amounts, in currency. Realised revenue.',
  pending_amount DECIMAL(28,2) COMMENT 'Sum of Pending transaction amounts, in currency.',
  refunded_amount DECIMAL(28,2) COMMENT 'Sum of Refunded transaction amounts, in currency.',
  subscription_amount DECIMAL(28,2) COMMENT 'Completed amount for product_type Subscription, in currency.',
  license_amount DECIMAL(28,2) COMMENT 'Completed amount for product_type License, in currency.',
  services_amount DECIMAL(28,2) COMMENT 'Completed amount for product_type Services, in currency.',
  support_amount DECIMAL(28,2) COMMENT 'Completed amount for product_type Support, in currency.',
  CONSTRAINT keys_present EXPECT (account_id IS NOT NULL AND currency IS NOT NULL AND transaction_date IS NOT NULL) ON VIOLATION FAIL UPDATE,
  CONSTRAINT account_found EXPECT (account_name IS NOT NULL),
  CONSTRAINT amounts_non_negative EXPECT (completed_amount >= 0 AND pending_amount >= 0 AND refunded_amount >= 0),
  CONSTRAINT product_split_equals_completed EXPECT (subscription_amount + license_amount + services_amount + support_amount = completed_amount),
  CONSTRAINT gold_revenue_by_account_day_pk PRIMARY KEY (account_id, currency, transaction_date)
)
CLUSTER BY AUTO
COMMENT 'Gold daily revenue: transaction amounts by status and product type per account, currency and transaction day. Amounts are in the original currency (see the currency column) and are never converted: do not sum them across currencies. Group or filter by currency before adding rows together. Source of revenue_by_account_month.'
AS
SELECT
  t.account_id, a.account_name, t.currency, t.transaction_date,
  trunc(t.transaction_date, 'MM') AS month,
  count(*) AS transaction_count,
  sum(CASE WHEN t.status = 'Completed' THEN 1 ELSE 0 END) AS completed_transaction_count,
  CAST(coalesce(sum(CASE WHEN t.status = 'Completed' THEN t.amount END), CAST(0 AS DECIMAL(18,2))) AS DECIMAL(28,2)) AS completed_amount,
  CAST(coalesce(sum(CASE WHEN t.status = 'Pending' THEN t.amount END), CAST(0 AS DECIMAL(18,2))) AS DECIMAL(28,2)) AS pending_amount,
  CAST(coalesce(sum(CASE WHEN t.status = 'Refunded' THEN t.amount END), CAST(0 AS DECIMAL(18,2))) AS DECIMAL(28,2)) AS refunded_amount,
  CAST(coalesce(sum(CASE WHEN t.status = 'Completed' AND t.product_type = 'Subscription' THEN t.amount END), CAST(0 AS DECIMAL(18,2))) AS DECIMAL(28,2)) AS subscription_amount,
  CAST(coalesce(sum(CASE WHEN t.status = 'Completed' AND t.product_type = 'License' THEN t.amount END), CAST(0 AS DECIMAL(18,2))) AS DECIMAL(28,2)) AS license_amount,
  CAST(coalesce(sum(CASE WHEN t.status = 'Completed' AND t.product_type = 'Services' THEN t.amount END), CAST(0 AS DECIMAL(18,2))) AS DECIMAL(28,2)) AS services_amount,
  CAST(coalesce(sum(CASE WHEN t.status = 'Completed' AND t.product_type = 'Support' THEN t.amount END), CAST(0 AS DECIMAL(18,2))) AS DECIMAL(28,2)) AS support_amount
FROM silver.sf_transactions t
LEFT JOIN silver.sf_accounts a ON a.account_id = t.account_id
GROUP BY t.account_id, a.account_name, t.currency, t.transaction_date;

In [0]:
-- revenue_by_account_month: monthly roll-up of revenue_by_account_day
CREATE OR REFRESH MATERIALIZED VIEW gold.revenue_by_account_month (
  account_id STRING NOT NULL COMMENT 'Account key (FK to account_360.account_id). Part of the primary key.',
  account_name STRING COMMENT 'Company name of the account.',
  currency STRING NOT NULL COMMENT 'ISO currency code of all amounts in the row (not converted). Part of the primary key.',
  month DATE NOT NULL COMMENT 'First day of the transaction calendar month. Part of the primary key.',
  active_days BIGINT COMMENT 'Number of days in the month with at least one transaction.',
  transaction_count BIGINT COMMENT 'Number of transactions in the period (all statuses).',
  completed_transaction_count BIGINT COMMENT 'Transactions with status Completed.',
  completed_amount DECIMAL(28,2) COMMENT 'Sum of Completed transaction amounts, in currency. Realised revenue.',
  pending_amount DECIMAL(28,2) COMMENT 'Sum of Pending transaction amounts, in currency.',
  refunded_amount DECIMAL(28,2) COMMENT 'Sum of Refunded transaction amounts, in currency.',
  subscription_amount DECIMAL(28,2) COMMENT 'Completed amount for product_type Subscription, in currency.',
  license_amount DECIMAL(28,2) COMMENT 'Completed amount for product_type License, in currency.',
  services_amount DECIMAL(28,2) COMMENT 'Completed amount for product_type Services, in currency.',
  support_amount DECIMAL(28,2) COMMENT 'Completed amount for product_type Support, in currency.',
  CONSTRAINT keys_present EXPECT (account_id IS NOT NULL AND currency IS NOT NULL AND month IS NOT NULL) ON VIOLATION FAIL UPDATE,
  CONSTRAINT account_found EXPECT (account_name IS NOT NULL),
  CONSTRAINT amounts_non_negative EXPECT (completed_amount >= 0 AND pending_amount >= 0 AND refunded_amount >= 0),
  CONSTRAINT product_split_equals_completed EXPECT (subscription_amount + license_amount + services_amount + support_amount = completed_amount),
  CONSTRAINT gold_revenue_by_account_month_pk PRIMARY KEY (account_id, currency, month)
)
CLUSTER BY AUTO
COMMENT 'Gold monthly revenue: roll-up of revenue_by_account_day per account, currency and calendar month (amounts by status and product type). Amounts are in the original currency (see the currency column) and are never converted: do not sum them across currencies. Group or filter by currency before adding rows together.'
AS
SELECT
  account_id, account_name, currency, month,
  count(*) AS active_days,
  sum(transaction_count) AS transaction_count,
  sum(completed_transaction_count) AS completed_transaction_count,
  CAST(sum(completed_amount) AS DECIMAL(28,2)) AS completed_amount,
  CAST(sum(pending_amount) AS DECIMAL(28,2)) AS pending_amount,
  CAST(sum(refunded_amount) AS DECIMAL(28,2)) AS refunded_amount,
  CAST(sum(subscription_amount) AS DECIMAL(28,2)) AS subscription_amount,
  CAST(sum(license_amount) AS DECIMAL(28,2)) AS license_amount,
  CAST(sum(services_amount) AS DECIMAL(28,2)) AS services_amount,
  CAST(sum(support_amount) AS DECIMAL(28,2)) AS support_amount
FROM gold.revenue_by_account_day
GROUP BY account_id, account_name, currency, month;